# Lab 4: ensembles

The same pipeline, trained on a proper score instead of the mean squared error, with a noise vector as a second input.
The model becomes a function of the state and the noise, one draw gives one member, and the score compares the members with the one observation.
Sections 1 to 3 build the three pieces: the score, the forward step that draws the members, and the noise input in the network.
Section 4 trains, and sections 5 to 7 verify: the members and their mean against the deterministic model, the information and noise decomposition of both, and the rank histogram.

This lab hands out no configuration files.
Start from your lab 3 `configs/vit_mse.yaml`, copy it once per run, and add the fields the sections below name; `configs/det_mse.yaml`, `configs/crps_add.yaml`, `configs/crps_concat.yaml`, and `configs/crps_adaln.yaml` are the names the cells here expect.
Conventions as in labs 2 and 3, with one axis added: the members sit on the second axis of a tensor, `(batch, member, variable, latitude, longitude)`, and on a dimension called `number` in xarray, which is WeatherBench 2's name for the ensemble member.
The runs are 2000 steps each and take a few minutes together on a laptop.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import torch
import lightning as L
from lightning.pytorch.loggers import CSVLogger

from utils.config import Config
from utils.lightning_module import ForecastModule, forecasts_to_xarray, LossCurve

xr.set_options(keep_attrs=True, display_expand_data=False, use_bottleneck=False)
torch.manual_seed(0)

DATA = Path("data")
LOGS, CHECKPOINTS, FORECASTS = Path("logs"), Path("checkpoints"), DATA / "forecasts"
for p in (LOGS, CHECKPOINTS, FORECASTS):
    p.mkdir(parents=True, exist_ok=True)

era5 = xr.open_zarr(DATA / "era5_5p6.zarr")             # the fields of labs 2 and 3
clim = xr.open_zarr(DATA / "clim_5p6.zarr").load()      # the training-years climatology of lab 3

### Documentation

- [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html): the noise, drawn white.
- [einops.repeat](https://einops.rocks/api/repeat/) and [einops.pack](https://einops.rocks/4-pack-and-unpack/): the member axis and the extra input channels.
- [xarray.Dataset.var](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.var.html): the ensemble spread.

## 1. The empirical score (`utils/loss_fn.py`)

The continuous ranked probability score of an ensemble is the lecture's expression: the mean absolute error of the members against the observation, less half the mean absolute difference between the members.
For $m$ members $x^{(1)}, \dots, x^{(m)}$ and the observation $y$,

$$\mathrm{CRPS} = \frac{1}{m}\sum_{i} \big|x^{(i)} - y\big| \;-\; \frac{1}{2m^2}\sum_{i}\sum_{j} \big|x^{(i)} - x^{(j)}\big|,$$

and the fair form divides the second sum by $2m(m-1)$ instead, which makes it unbiased in the ensemble size, so four members already estimate the score of the distribution the model represents rather than the score of a four-member sample.

- `EmpiricalCRPS(fair=True)`: a `torch.nn.Module` like `MSE`.
  `forward(prediction, target)` takes the members on the second axis, `(b, m, v, h, w)`, and the target `(b, v, h, w)`, and returns the mean of the score over every axis.
  Write the score itself as a function of an array with the members last, `(..., m)`, and move the axis with one `rearrange` in `forward`; the pairwise sum runs over all $m^2$ ordered pairs, whose diagonal contributes nothing.
- `ObjectiveConfig`: `name` gains `"crps"`, and `kwargs` carries `fair`.
- `ForecastModule.__init__` builds it by name, as in lab 3; the validation loss stays the plain `MSE` of the ensemble mean, and the score of the members is logged next to it, so the runs of this lab and the runs of lab 3 compare on one axis.

The double sum costs $m^2$ absolute differences per grid point, which is cheap at four members; on the sorted members the same quantity costs $m$, and that is the form a production implementation uses.

Checks: an ensemble whose members all equal the observation scores exactly 0; with one member the score is the absolute error, and `fair=True` is refused; the fair form is smaller than the unfair one on the same members, because it subtracts more; both agree with the double sum written out with two loops on random numbers.

## 2. The forward step (`utils/lightning_module.py`)

One state, several members: the state is repeated along a new axis, one noise vector is drawn per member, the network runs once on the whole batch, and the members come back on their own axis.
This is the lecture listing.

- `TrainerConfig` gains `ens_size` (the members per sample in the training step, 4) and `eval_ens_size` (the members per initialisation in the validation and prediction steps, 8).
- `ForecastModule.forward_step(state, members, time=None)`: `repeat` the state `(b, v, H, W)` to `(b m, v, H, W)`, draw `torch.randn(b * m, dim_noise)` on the state's device, call the network with the state, the time, and the noise, and `rearrange` the output back to `(b, m, v, H, W)`.
- `forecast(x, steps, time=None, members=1)` returns `(b, m, v, steps, H, W)`: the members are drawn at the first step, and after it each member carries on from its own state with a fresh noise draw at every step, so a roll-out samples one path per member rather than repeating one draw.
- `training_step`: the forecast of `train_rollout_steps` steps with `ens_size` members, and the objective of each step against its target; a score takes the member axis, `MSE` takes the single member.
- `validation_step` and `predict_step` use `eval_ens_size` members.
  The validation logs the mean squared error of the ensemble mean of step $k$ as `val/loss_step{k}`, as in lab 3, and the score of its members as `val/crps_step{k}` beside it.
  `predict_step` returns the roll-out of `predict_steps` on the CPU, `(b, m, v, predict_steps, H, W)`.
- `forecasts_to_xarray` gains the member axis: `number` as the first named axis of `to_xarray`, so a forecast Dataset has the dimensions `(number, time, prediction_timedelta, latitude, longitude)`, and lab 3's `rearrange` of every batch carries the member axis with it.
  A deterministic run goes through the same path with one member, so its forecast keeps a `number` dimension of length one.

Checks: `forward_step` on a batch of three states with four members returns `(3, 4, 9, 32, 64)` and the members differ; the roll-out of three steps with four members returns `(3, 4, 9, 3, 32, 64)`; a deterministic run through the same path keeps a `number` dimension of length one; a network without a noise input gives identical members.

## 3. The noise in the model (`utils/components.py`)

The network takes a second input, the noise vector `(b, dim_noise)`, and becomes a function of the state and the noise.
Implement one of the three ways in; each is a config field rather than a rewrite, so a second and a third cost only another configuration file.

- `NetworkConfig` gains `noise_injection` (`"none"`, `"add"`, `"concat"`, or `"adaln"`) and `dim_noise` (32).
- `ViT.forward(x, time=None, noise=None)` with
  - `"add"`: a bias-free linear map from `dim_noise` to the token width, followed by an `RMSNorm`, repeated over the tokens and added to them after the patch embedding and the positions.
  - `"concat"`: the noise broadcast from `(b, dim_noise)` to `dim_noise` constant fields and packed with the state on the channel axis before the patch embedding, so the embedding takes `num_variables + dim_noise` input channels while the unembedding still writes `num_variables`.
  - `"adaln"`: the noise as the context of an adaptive layer normalisation in every block.
    `AdaptiveLayerNorm(dim, dim_ctx=None)`: a `LayerNorm` without its own affine, and one linear map from the context to a scale and a shift, applied as `(1 + scale) * norm(x) + shift`.
    Initialise its weight from a truncated normal at `1e-4` and its bias at zero, so the block starts close to a plain normalisation and the context still carries a gradient from the first step; a zero weight starts with none and converges slowly.
    `TransformerBlock` takes `dim_ctx` and passes the context to both norms; without it the block is lab 2's.
- `Persistence.forward` takes the noise and ignores it.

Checks: the network maps a state to a state of the same shape under every setting; two different noise vectors give different outputs, and the same noise vector gives the same output twice; `"add"` costs `dim_noise * width` weights for the map and `width` for the norm against the plain network; with `noise_injection: none` the noise makes no difference at all.

## 4. Train

`configs/crps_add.yaml` (or `crps_concat.yaml`, or `crps_adaln.yaml`) is lab 3's `vit_mse.yaml` with the fields of sections 1 to 3 set: `objective.name: crps` with `fair: true`, `network.noise_injection` and `dim_noise: 32`, `trainer.ens_size: 4`, `eval_ens_size: 8`, and `dataset.batch_size: 4`.
Four members at a quarter of the batch size hold the number of states per step at sixteen.
A training step therefore costs what lab 3's costs, and the two runs are read against each other at equal cost.
What the run gives up is the number of distinct initial conditions per step.
`configs/det_mse.yaml` is the same file with `ens_size: 1`, `noise_injection: none`, the mean squared error, and `batch_size: 16`: the deterministic baseline of every score below.
`dim_noise` stays set in it, since the forward step draws the noise whether or not the network reads it.
Both carry the prediction settings of this lab, `predict_steps: 20` and `predict_stride: 28`, which is one initialisation a week over 2019, 52 of them at 00 UTC.

In [ ]:
def train(run: str) -> xr.Dataset:
    '''Train one configuration, save its checkpoint, and write its forecasts of the evaluation year to disk.'''
    config = Config.from_yaml(f"configs/{run}.yaml")
    module = ForecastModule(config)
    trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=run), log_every_n_steps=10,
                        enable_checkpointing=False, val_check_interval=200, limit_val_batches=10, callbacks=[LossCurve()])
    trainer.fit(module)
    trainer.save_checkpoint(CHECKPOINTS / f"{run}.ckpt")
    return forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{run}.zarr")

det_forecast = train("det_mse")
crps_forecast = train("crps_add")
crps_forecast

The two training losses are two different objectives and do not compare; the validation curves do, `val/loss_step1` being the mean squared error of the ensemble mean in both runs.
The score of the members, `val/crps_step1`, is logged for the ensemble run alone.

## 5. Scores of the members and of their mean (`utils/metrics.py`)

The verification of lab 1 applies to every member as it stands, so the only new step is where the mean over `number` is taken.

- `ensemble_mean(forecast)`: the mean over `number`, and a forecast without that dimension (lab 3 wrote none) unchanged.
- `acc(forecast, truth, clim)`: the anomaly correlation of lab 1, section 5.4, per initialisation, averaged over the initialisations; anomalies against the climatology at the valid time.
- The truth is lab 3's `truth_at(era5, forecast)` on a forecast with the member axis dropped, `forecast.isel(number=0, drop=True)`, so that it has the initialisation and lead dimensions and no member axis; it broadcasts against the members.
- Compute four curves per variable: the RMSE of the ensemble mean, the RMSE of a member (score every member and average the scores, not the other way round), the ACC of the ensemble mean, and the ACC of a member.
  Add the deterministic run to both panels.

Which of the two RMSE curves is lower and by how much, whether the same ordering holds for the ACC, and where the deterministic model sits against each, are what to read off.
A member is a draw from the forecast distribution and the mean is not: the mean of several draws is smoother than any of them.

Checks: the ensemble mean scores at least as well as the average member at every lead; with one member the two curves coincide; the ACC lies between -1 and 1 everywhere.

## 6. The information and noise decomposition (`utils/metrics.py`)

Lab 1, section 5.7, split the error of a forecast into the part along the true anomaly and the part orthogonal to it; the helper that draws the diagram is in that notebook, and the definitions are $p = a_f\,\mathrm{ACC}$, $\mathrm{IE} = |a_t - p|$, and $\mathrm{NE} = \sqrt{\mathrm{RMSE}^2 - \mathrm{IE}^2}$.

- `activity(anomalies)`: the area-weighted root mean square of an anomaly field over the grid, averaged over the initialisations, which is the $a_f$ and $a_t$ of lab 1.
- Compute the activity, the ACC, and the RMSE of one member and of the ensemble mean, and from them the information, the information error, and the noise error at every lead.
- Move the diagram helper of lab 1 into `utils/metrics.py`, so that both notebooks call the same one.
- Draw both on one diagram, with the deterministic run beside them.

Where the member sits against the mean, which of the two errors separates them and which does not, and how each moves as the lead grows, are what to read off.

Checks: the information equals the activity times the ACC; the squared information error and the squared noise error sum to the squared RMSE; a member's activity is close to the true activity while the mean's falls below it.

## 7. The rank histogram (`utils/metrics.py`)

The rank of an observation in an ensemble is the number of members below it, so with $m$ members there are $m + 1$ ranks.
If the observation is a draw from the same distribution as the members, every rank is equally likely and the histogram is flat at $1/(m+1)$.
A U shape means the observation falls outside the members too often, an ensemble too narrow for its error; a dome means the opposite.
The implementation is given, on one variable at a time: every grid point of every initialisation is one observation, and the ranks are counted with numpy rather than through xarray.

In [ ]:
def rank_histogram(ensemble, truth):
    '''The frequency of each rank of the observation among the members: m + 1 bins summing to one.'''
    m = ensemble.sizes["number"]
    members = ensemble.transpose(..., "number").values.reshape(-1, m)
    observation = truth.values.reshape(-1, 1)
    valid = ~np.isnan(members).any(-1) & ~np.isnan(observation[:, 0])
    members, observation = members[valid], observation[valid]
    return np.bincount(np.sum(members < observation, axis=-1), minlength=m + 1) / len(members)


def plot_rank_histogram(ax, ensemble, truth, leads, labels=None):
    '''One group of bars per lead time, and the flat line at 1 / (m + 1) that a calibrated ensemble follows.'''
    m, n = ensemble.sizes["number"], len(leads)
    width, bins = 0.8 / n, np.arange(m + 1)
    colours = plt.colormaps["viridis"](np.linspace(0.2, 0.85, n))
    for i, lead in enumerate(leads):
        ranks = rank_histogram(ensemble.sel(prediction_timedelta=lead), truth.sel(prediction_timedelta=lead))
        ax.bar(bins + i * width, ranks, width=width, alpha=0.8, color=colours[i], edgecolor="k", lw=0.4,
               label=labels[i] if labels is not None else str(lead))
    ax.axhline(1 / (m + 1), color="red", linestyle="dashed", lw=1)
    ax.set_xticks(bins + width * (n - 1) / 2, bins)
    ax.set_xlabel("rank of the observation")
    ax.set_ylabel("frequency")
    ax.legend()
    return ax

Plot it for `crps_forecast["Z500"]` and `crps_forecast["T2M"]` against the truth at 6 hours, 1 day, 3 days, and 5 days, and read the shape at each lead.
Then the diagnostic the shape summarises: the spread-skill ratio, the ensemble spread over the error of the ensemble mean,

$$\mathrm{SSR} = \sqrt{\frac{m+1}{m}} \; \frac{\sqrt{\langle \mathrm{var}_{\text{number}} f \rangle_w}}{\sqrt{\langle (t - \bar f)^2 \rangle_w}},$$

with the factor in front correcting for the finite ensemble; a calibrated ensemble sits at one, below one is under-dispersive and above one over-dispersive.
Plot it against lead time next to the histograms, and say which way this ensemble is miscalibrated and whether it gets better or worse with lead time.

Checks: the histogram has `eval_ens_size + 1` bins and sums to one; an ensemble whose members are all equal puts every observation in an end bin; the ratio is positive and finite at every lead.

## Extension: the other two injections

Train `crps_concat` and `crps_adaln` from their configuration files and put all three on the plots of sections 5 to 7.
The three differ in where the noise enters, not in what they are trained on, so the comparison is of the injection alone.